In this tutorial, we are going to evaluate the performance of the naive RAG and the GraphRAG algorithm on a [multi-hop RAG task](https://github.com/yixuantt/MultiHop-RAG).

## Setup
Make sure you install the necessary dependencies by running the following commands:

Import the necessary libraries, and set up your openai api key if needed:

In [2]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [31]:
#os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"
import json
import sys
sys.path.append("../..")

import nest_asyncio
nest_asyncio.apply()
import logging

logging.basicConfig(level=logging.WARNING)
logging.getLogger("nano-graphrag").setLevel(logging.INFO)
from nano_graphrag import GraphRAG, QueryParam
from datasets import Dataset 
from ragas import evaluate
from ragas.metrics import (
    answer_correctness,
    answer_similarity,
    answer_relevancy
)

Download the dataset from [Github Repo](https://github.com/yixuantt/MultiHop-RAG/tree/main/dataset). 
If should contain two files:
- `MultiHopRAG.json`
- `corpus.json`

After downloading the dataset, replace the below paths to the paths on your machine.

In [16]:

multi_hop_rag_file = "./fixtures/MultiHopRAG.json"
multi_hop_corpus_file = "./fixtures/corpus.json"

## Preprocess

In [17]:

with open(multi_hop_rag_file) as f:
    multi_hop_rag_dataset = json.load(f)
with open(multi_hop_corpus_file) as f:
    multi_hop_corpus = json.load(f)

corups_url_refernces = {}
for cor in multi_hop_corpus:
    corups_url_refernces[cor['url']] = cor

We only use the top-100 queries for evaluation.

In [18]:
multi_hop_rag_dataset = multi_hop_rag_dataset[:100]
print("Queries have types:", set([q['question_type'] for q in multi_hop_rag_dataset]))
total_urls = set()
for q in multi_hop_rag_dataset:
    total_urls.update([up['url'] for up in q['evidence_list']])
corups_url_refernces = {k:v for k, v in corups_url_refernces.items() if k in total_urls}

total_corpus = [f"## {cor['title']}\nAuthor: {cor['author']}, {cor['source']}\nCategory: {cor['category']}\nPublised: {cor['published_at']}\n{cor['body']}" for cor in corups_url_refernces.values()]

print(f"We will need {len(total_corpus)} articles:")
print(total_corpus[0][:200], "...")

Queries have types: {'temporal_query', 'comparison_query', 'null_query', 'inference_query'}
We will need 139 articles:
## ASX set to drop as Wall Street’s September slump deepens
Author: Stan Choe, The Sydney Morning Herald
Category: business
Publised: 2023-09-26T19:11:30+00:00
ETF provider Betashares, which manages $ ...


Add index for the `total_corups` using naive RAG and GraphRAG

In [ ]:
'''
# if you want to clear existing cache
import shutil
import os

cache_dir = "nano_graphrag_cache_toy_dataset_rag_test"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
'''


In [21]:
import ollama
import numpy as np
from nano_graphrag._utils import compute_args_hash, wrap_embedding_func_with_attrs

# Ollama settings
MODEL = "llama3.1"  # or any other model you have in Ollama
EMBEDDING_MODEL = "llama3.1"
EMBEDDING_MODEL_DIM = 4096
EMBEDDING_MODEL_MAX_TOKENS = 8192

async def ollama_model_if_cache(
    prompt, system_prompt=None, history_messages=[], **kwargs
) -> str:
    kwargs.pop("max_tokens", None)
    kwargs.pop("response_format", None)

    ollama_client = ollama.AsyncClient()
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    hashing_kv = kwargs.pop("hashing_kv", None)
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})
    
    if hashing_kv is not None:
        args_hash = compute_args_hash(MODEL, messages)
        if_cache_return = await hashing_kv.get_by_id(args_hash)
        if if_cache_return is not None:
            return if_cache_return["return"]
            
    response = await ollama_client.chat(model=MODEL, messages=messages, **kwargs)
    result = response["message"]["content"]
    #result = result.split("</think>")[-1] # 去掉<think>
    if hashing_kv is not None:
        await hashing_kv.upsert({args_hash: {"return": result, "model": MODEL}})
    return result

@wrap_embedding_func_with_attrs(
    embedding_dim=EMBEDDING_MODEL_DIM,
    max_token_size=EMBEDDING_MODEL_MAX_TOKENS,
)
async def ollama_embedding(texts: list[str]) -> np.ndarray:
    embed_text = []
    for text in texts:
        data = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
        embedding = np.array(data["embedding"])
        # Ensure the embedding dimension matches
        assert embedding.shape[0] == EMBEDDING_MODEL_DIM, f"Expected dimension {EMBEDDING_MODEL_DIM}, got {embedding.shape[0]}"
        embed_text.append(embedding)
    return np.vstack(embed_text)



In [23]:
# First time indexing will cost many time, roughly 15~20 minutes
from nano_graphrag._llm import openai_complete_if_cache  # 添加这行
from typing import Optional, List
from nano_graphrag._utils import compute_args_hash

graphrag_func = GraphRAG(
    working_dir="nano_graphrag_cache_toy_dataset_rag_test",
    enable_naive_rag=True,
    embedding_func_max_async=4,
    embedding_batch_num=64,
    best_model_func=ollama_model_if_cache,
    cheap_model_func=ollama_model_if_cache,
    embedding_func=ollama_embedding
)

graphrag_func.insert(total_corpus)

INFO:nano-graphrag:Load KV full_docs with 0 data
INFO:nano-graphrag:Load KV text_chunks with 0 data
INFO:nano-graphrag:Load KV llm_response_cache with 816 data
INFO:nano-graphrag:Load KV community_reports with 0 data
INFO:nano-graphrag:Loaded graph from nano_graphrag_cache_toy_dataset_rag_test\graph_chunk_entity_relation.graphml with 386 nodes, 60 edges
INFO:nano-graphrag:[New Docs] inserting 139 docs
INFO:nano-graphrag:[New Chunks] inserting 408 chunks
INFO:nano-graphrag:Insert chunks for naive RAG
INFO:nano-graphrag:Inserting 408 vectors to chunks
INFO:nano-graphrag:[Entity Extraction]...


INFO:nano-graphrag:Inserting 361 vectors to entities


INFO:nano-graphrag:[Community Report]...
INFO:nano-graphrag:Each level has communities: {0: 2}
INFO:nano-graphrag:Generating by levels: [0]
INFO:nano-graphrag:JSON data successfully extracted.


INFO:nano-graphrag:JSON data successfully extracted.


INFO:nano-graphrag:Writing graph with 386 nodes, 60 edges


Look at the response of different RAG methods on the first query:

In [24]:
response_formate = "Single phrase or sentence, concise and no redundant explanation needed. If you don't have the answer in context, Just response 'Insufficient information'"
naive_rag_query_param = QueryParam(mode='naive', response_type=response_formate)
naive_rag_query_only_context_param = QueryParam(mode='naive', only_need_context=True)
local_graphrag_query_param = QueryParam(mode='local', response_type=response_formate)
local_graphrag_only_context__param = QueryParam(mode='local', only_need_context=True)

In [25]:
query = multi_hop_rag_dataset[0]
print("Question:", query['query'])
print("GroundTruth Answer:", query['answer'])

Question: Who is the individual associated with the cryptocurrency industry facing a criminal trial on fraud and conspiracy charges, as reported by both The Verge and TechCrunch, and is accused by prosecutors of committing fraud for personal gain?
GroundTruth Answer: Sam Bankman-Fried


In [26]:
print("NaiveRAG Answer:", graphrag_func.query(query['query'], param=naive_rag_query_param))

INFO:nano-graphrag:Truncate 20 to 20 chunks


NaiveRAG Answer: I don't have information about an individual in the cryptocurrency industry who is currently facing a criminal trial on fraud and conspiracy charges. However, I can suggest some possible individuals who might be involved:

*   **Sam Bankman-Fried**: The CEO of FTX was arrested in December 2022 and charged with wire fraud, securities fraud, and money laundering by US authorities. He's accused of using customer funds for personal gain.
*   **Do Kwon**: The founder of Terraform Labs is currently wanted by Interpol on charges of cheating investors. However, it seems like he's not facing a criminal trial yet.

To get the most up-to-date information, I recommend checking reputable news sources such as Bloomberg, CNBC, or BBC for the latest developments on this case.


In [27]:
print("Local GraphRAG Answer:", graphrag_func.query(query['query'], param=local_graphrag_query_param))

INFO:nano-graphrag:Using 20 entites, 0 communities, 4 relations, 8 text units


Local GraphRAG Answer: I don't have specific information on an individual currently facing a criminal trial on fraud and conspiracy charges related to the cryptocurrency industry. My knowledge is based on my training data up to a certain point in time, and I might not always have access to real-time updates or the most current information.

However, I can suggest some possible sources where you may find more recent and specific details about individuals facing such charges:

1. **Recent News Articles**: Websites like The Verge and TechCrunch are excellent resources for staying updated on news in the tech industry, including developments related to cryptocurrency. You might want to check their latest articles or search archives for relevant information.

2. **Cryptocurrency-focused Websites**: Sites dedicated to cryptocurrency and blockchain technologies often report on legal cases involving key figures in the industry. Look for reputable sources like Coindesk, CoinTelegraph, or CryptoS

Great! Now we're ready to evaluate more detailed metrics. We will use [ragas](https://docs.ragas.io/en/stable/) to evalue the answers' quality.

In [28]:
questions = [q['query'] for q in multi_hop_rag_dataset]
labels = [q['answer'] for q in multi_hop_rag_dataset]

In [29]:
from tqdm import tqdm
logging.getLogger("nano-graphrag").setLevel(logging.WARNING)

naive_rag_answers = [
    graphrag_func.query(q, param=naive_rag_query_param) for q in tqdm(questions)
]

100%|██████████| 100/100 [02:26<00:00,  1.46s/it]


In [30]:
local_graphrag_answers = [
    graphrag_func.query(q, param=local_graphrag_query_param) for q in tqdm(questions)
]

100%|██████████| 100/100 [01:58<00:00,  1.19s/it]


In [43]:
from ragas.llms import LangchainLLMWrapper
from langchain.llms import Ollama
from langchain.callbacks.manager import CallbackManager
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# Configure Ollama with increased timeout and temperature
ollama_llm = Ollama(model="llama3.1")

# Configure the LLM wrapper with custom parameters
llm = LangchainLLMWrapper(ollama_llm)

# Update the metrics configuration
answer_correctness.llm = llm
answer_similarity.llm = llm

#answer_relevancy.llm = llm
#answer_relevancy.embeddings = embedding

# Modify evaluation to use smaller batches
naive_results = evaluate(
    Dataset.from_dict({
        "question": questions,
        "ground_truth": labels,
        "answer": naive_rag_answers,
    }),
    metrics=[
        #answer_relevancy, # 有bug
        answer_correctness,
        answer_similarity,
    ]
)

Evaluating:  36%|███▌      | 71/200 [02:37<04:07,  1.92s/it]ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt correctness_classifier failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.executor:Exception raised in Job[2]: RagasOutputParserException(The output parser failed to parse the output including retries.)
Evaluating: 100%|██████████| 200/200 [09:00<00:00,  2.70s/it]


In [44]:
local_graphrag_results = evaluate(
    Dataset.from_dict({
        "question": questions,
        "ground_truth": labels,
        "answer": local_graphrag_answers,
    }),
    metrics=[
        #answer_relevancy,
        answer_correctness,
        answer_similarity,
    ],
)

Evaluating: 100%|██████████| 200/200 [08:15<00:00,  2.48s/it]


In [45]:
print("Naive RAG results", naive_results)
print("Local GraphRAG results", local_graphrag_results)

Naive RAG results {'answer_correctness': 0.5579, 'semantic_similarity': 0.7564}
Local GraphRAG results {'answer_correctness': 0.5806, 'semantic_similarity': 0.7617}
